# 9.1 Stage 1: Stabilization (Expanded)

Holds current pose using PD control.

---

```python id="0c0d5y"
if t < d:
    self.log_stage(1, "Stabilizing (holding current pose)")

    for j in self.joints:
        q = self.low_state.motor_state[j].q
        self.low_cmd.motor_cmd[j].q = q
        self.low_cmd.motor_cmd[j].dq = 0
        self.low_cmd.motor_cmd[j].kp = self.kp
        self.low_cmd.motor_cmd[j].kd = self.kd
        self.low_cmd.motor_cmd[j].tau = 0
```

---

## 🧠 Big Picture: What Is This Stage Doing?

This stage tells the robot:

```text id="7z9dbw"
"Stay exactly where you are"
```

---

But importantly:

> It does NOT turn motors off
> It actively **holds the current pose using feedback control**

---

## 🔄 Control Strategy

For each joint:

```text id="lccqbw"
1. Read current position (q)
2. Set target position = current position
3. Apply PD control
```

---

## 🧱 Step-by-Step Breakdown

---

### 🔹 Step 1: Read Current Position

```python id="nq3r2k"
q = self.low_state.motor_state[j].q
```

---

### Meaning:

```text id="t1f6k3"
q = actual joint angle right now
```

---

### 🧠 This is feedback

You are measuring the system state before acting.

---

## 🔹 Step 2: Set Target = Current

```python id="6z5v9c"
self.low_cmd.motor_cmd[j].q = q
```

---

### This creates:

```text id="m2kn8n"
q_target = q_actual
```

---

### Resulting error:

[
q_{target} - q = 0
]

---

## 🔹 Step 3: Set Velocity Target

```python id="l0qsnl"
self.low_cmd.motor_cmd[j].dq = 0
```

---

### Meaning:

```text id="o6x6k3"
Do not move (zero desired velocity)
```

---

## 🔹 Step 4: Apply PD Gains

```python id="6nd5oy"
self.low_cmd.motor_cmd[j].kp = self.kp
self.low_cmd.motor_cmd[j].kd = self.kd
```

---

### This activates the controller:

[
\tau = K_p (q_{target} - q) + K_d (0 - \dot{q})
]

---

### Since:

[
q_{target} = q
]

---

### The equation reduces to:

[
\tau = -K_d \dot{q}
]

---

## 🔬 Interpretation

This stage becomes:

> 🎯 **Pure damping control**

---

### Meaning:

* If joint is moving → resist motion
* If joint is still → apply zero torque

---

## 🔹 Step 5: No Feedforward Torque

```python id="9l3c9u"
self.low_cmd.motor_cmd[j].tau = 0
```

---

### Meaning:

```text id="q0p2i3"
No additional torque is applied
```

---

## 🧠 Physical Interpretation

This stage behaves like:

```text id="k89xqg"
A robotic arm floating in a viscous fluid
```

---

* If you push it → it resists
* If you let go → it stays still

---

## ⚠️ Important Reality

Even though:

```text id="g8bb3o"
error = 0
```

The robot still needs torque to:

* Counter gravity
* Maintain posture
* Resist disturbances

---

### Where does that come from?

From the **internal motor control + feedback loop**.

---

## 🔄 Why This Stage Exists

### 1. Safe Startup

Before moving:

```text id="kp6m6l"
Stabilize robot in its current configuration
```

---

### 2. Eliminate Residual Motion

If robot is slightly moving:

```text id="j66iqt"
Damping will bring it to rest
```

---

### 3. Synchronization

Ensures:

```text id="qj1tv1"
Controller state = physical state
```

---

## ⚙️ Control Theory View

This stage implements:

> A **zero-error equilibrium condition**

---

### System is driven to:

[
\dot{q} = 0
]

---

## 🧠 Subtle but Critical Insight

You are NOT doing:

```python
cmd.q = constant_value
```

You are doing:

```python
cmd.q = measured_value
```

---

### This difference is HUGE:

| Approach        | Behavior                   |
| --------------- | -------------------------- |
| Fixed target    | Robot moves to that pose   |
| Measured target | Robot holds current pose ✅ |

---

## 🔄 Loop Behavior

Since this runs every 20 ms:

```text id="m6c8sl"
Continuously updates target to current state
```

---

### This creates:

> A **self-stabilizing system**

---

## ⚠️ What If You Removed This Stage?

Then:

* Stage 2 would start immediately
* Robot may:

  * Jerk
  * Overshoot
  * Be unstable

---

## 🤖 RL Interpretation

This stage is equivalent to:

```text id="zq7j7o"
"Reset stabilization phase"
```

---

In RL environments:

* Often you:

  * Reset state
  * Wait for stabilization
  * THEN start actions

---

## 🔥 Hidden Engineering Insight

This stage is doing something very important:

> It **conditions the system before applying motion**

---

### In real robotics:

This is often called:

* Settling phase
* Hold phase
* Pre-control stabilization

---

## 🔄 Analogy

Think of this like:

```text id="slc03s"
Holding your arm still before starting a movement
```

---

You don’t:

* Start moving while unstable

You:

* Stabilize first
* Then move

---

## 🚀 Summary

This stage:

| Step               | Role                 |
| ------------------ | -------------------- |
| Read `q`           | Get current position |
| Set `q_target = q` | Zero position error  |
| Apply `kp`, `kd`   | Activate control     |
| Use `dq = 0`       | Stop motion          |
| Use `tau = 0`      | No feedforward       |

---

### Result:

```text id="n1c7g9"
Robot holds its current pose safely and stably
```

---

> 🔥 This is the **foundation of stable motion**—everything that follows depends on this stage working correctly.

